# 05 · RLHF Fine-Tuning
**Owner: Wynnian** | Target: complete by Apr 7

Fine-tunes the base PPO agent with persona-specific reward models.
For each persona (conservative, balanced, aggressive), loads the base agent,
wraps the env with `RLHFRewardWrapper`, and fine-tunes for 300k steps.

**Input:** `base_agent_seed1.zip`, `reward_model_{persona}.pt`, `{persona}_norm_stats.npz`  
**Output:** `rlhf_agent_{conservative,balanced,aggressive}.zip`

In [2]:
# ── Imports ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback

from src.envs import make_env, DOW30_TICKERS
from src.reward_model import RewardModel, load_reward_model, FEATURE_KEYS
from src.metrics import sharpe_ratio

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

PyTorch: 2.10.0+cpu
CUDA available: False


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
# ── Load data (same as 03_base_training) ──────────────────────────────────
FEATURE_NAMES = [
    'close', 'close_norm', 'volume', 'close_1d_ret', 'close_5d_ret', 'close_20d_ret',
    'vol_20d', 'vol_60d', 'macd', 'rsi_14', 'volume_ratio',
]

def load_long(path):
    """Load parquet (wide) and convert to long format for FinRL."""
    df_wide = pd.read_parquet(path)
    pieces = []
    for tic in DOW30_TICKERS:
        cols = [f'{tic}_{feat}' for feat in FEATURE_NAMES]
        tmp = df_wide[cols].copy()
        tmp.columns = FEATURE_NAMES
        tmp['date'] = df_wide.index
        tmp['tic']  = tic
        pieces.append(tmp)
    df = pd.concat(pieces, axis=0, ignore_index=True)
    df = df[['date', 'tic'] + FEATURE_NAMES]
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['date', 'tic']).reset_index(drop=True)
    df.index = df['date'].factorize()[0]
    return df

print('Loading data from Drive...')
df_train = load_long(f'{DATA_DIR}/features_train.parquet')
df_val   = load_long(f'{DATA_DIR}/features_val.parquet')
print(f'Train: {df_train.shape} | Val: {df_val.shape}')
print(f'Train dates: {df_train["date"].min().date()} → {df_train["date"].max().date()}')
print(f'Val dates:   {df_val["date"].min().date()} → {df_val["date"].max().date()}')

Loading data from Drive...
Train: (60420, 12) | Val: (3720, 12)
Train dates: 2015-01-02 → 2022-12-30
Val dates:   2023-01-03 → 2023-06-30


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
# ── Load reward models + normalization stats ─────────────────────────────
# The reward models were trained with normalized features (z-score).
# Raw MLP outputs have arbitrary negative bias (Bradley-Terry loss only learns
# relative ordering, not absolute scale). We auto-calibrate by loading a
# per-persona "center" (mean raw score over training data) computed in 04,
# then apply tanh((raw - center) * 0.1) to produce scores in [-1, 1].

FEATURE_KEYS = ['annualized_return', 'sharpe', 'max_drawdown', 'volatility', 'calmar', 'turnover']

class NormalizedRewardModel:
    """Wraps a RewardModel with input normalization + output calibration."""
    def __init__(self, model, norm_stats_path):
        self.model = model
        stats = np.load(norm_stats_path)
        self.mean   = stats['mean']
        self.std    = stats['std']
        self.center = float(stats['center'][0])

    def score(self, summary_dict):
        """Score a trajectory summary dict → calibrated value in ~[-1, 1]."""
        features = np.array([summary_dict[k] for k in FEATURE_KEYS])
        features = np.nan_to_num(features, nan=0.0, posinf=10.0, neginf=-10.0)
        features_norm = (features - self.mean) / self.std
        features_norm = np.clip(features_norm, -5, 5)
        x = torch.tensor(features_norm.reshape(1, -1), dtype=torch.float32)
        self.model.eval()
        with torch.no_grad():
            raw_score = self.model(x).item()
        return float(np.tanh((raw_score - self.center) * 0.1))

personas = ['conservative', 'balanced', 'aggressive']
reward_models = {}

for persona in personas:
    model = load_reward_model(f'{CKPT_DIR}/reward_model_{persona}.pt')
    norm_path = f'{CKPT_DIR}/{persona}_norm_stats.npz'
    reward_models[persona] = NormalizedRewardModel(model, norm_path)

    # Quick sanity check with contrasting profiles
    low_risk  = {'annualized_return': 0.05, 'sharpe': 0.8, 'max_drawdown': 0.03,
                 'volatility': 0.08, 'calmar': 1.5, 'turnover': 0.3}
    high_risk = {'annualized_return': 0.40, 'sharpe': 1.2, 'max_drawdown': 0.35,
                 'volatility': 0.30, 'calmar': 1.1, 'turnover': 0.8}
    s_low  = reward_models[persona].score(low_risk)
    s_high = reward_models[persona].score(high_risk)
    print(f'{persona:15s} center={reward_models[persona].center:+.4f}  '
          f'low_risk={s_low:+.4f}  high_risk={s_high:+.4f}  '
          f'delta={s_high - s_low:+.4f}')

print('\nAll 3 reward models loaded with calibrated normalization.')

conservative    center=+0.0254  low_risk=-0.4247  high_risk=-0.9608  delta=-0.5362
balanced        center=-4.6379  low_risk=-0.5769  high_risk=-0.4943  delta=+0.0826
aggressive      center=-3.4598  low_risk=-0.4530  high_risk=+0.1976  delta=+0.6507

All 3 reward models loaded with calibrated normalization.


In [5]:
# ── Verify base agent loads correctly ────────────────────────────────────
# Use seed 2 (best of 3 seeds from base PPO training)
BASE_AGENT_PATH = f'{CKPT_DIR}/base_agent_seed2.zip'

test_env = make_env(df_train, mode='train', seed=42)
test_model = PPO.load(BASE_AGENT_PATH, env=test_env)
print(f'Base agent loaded: {BASE_AGENT_PATH}')
print(f'Policy: {test_model.policy}')
del test_model, test_env

/usr/local/lib/python3.12/dist-packages/websockets/legacy/__init__.py:6: DeprecationWarning: websockets.legacy is deprecated; see https://websockets.readthedocs.io/en/stable/howto/upgrade.html for upgrade instructions
  warnings.warn(  # deprecated in 14.0 - 2024-11-09
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Base agent loaded: /content/drive/MyDrive/3001_RL_group_project/Project/results/checkpoints/base_agent_seed2.zip
Policy: ActorCriticPolicy(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (pi_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (vf_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (mlp_extractor): MlpExtractor(
    (policy_net): Sequential(
      (0): Linear(in_features=361, out_features=256, bias=True)
      (1): Tanh()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): Tanh()
    )
    (value_net): Sequential(
      (0): Linear(in_features=361, out_features=256, bias=True)
      (1): Tanh()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): Tanh()
    )
  )
  (action_net): Linear(in_features=256, out_features=30, bias=True)
  (value_net): Linear(in_features=256, out_features=1, bias=True)
)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
# ── RLHF fine-tuning config ──────────────────────────────────────────────
RLHF_TIMESTEPS = 300_000    # per plan
RLHF_LAMBDA    = 0.5        # 50% base reward + 50% RLHF reward
EVAL_FREQ      = 10_000

print(f'RLHF timesteps per persona: {RLHF_TIMESTEPS:,}')
print(f'RLHF lambda: {RLHF_LAMBDA}')
print(f'Eval frequency: every {EVAL_FREQ:,} steps')

RLHF timesteps per persona: 300,000
RLHF lambda: 0.5
Eval frequency: every 10,000 steps


In [7]:
# ── Eval callback (reused from 03, adapted for RLHF) ─────────────────────
class RLHFEvalCallback(BaseCallback):
    """
    Evaluates RLHF agent on val env every eval_freq steps.
    Saves checkpoint when val Sharpe improves.
    Also logs base_reward and rlhf_reward separately.
    """
    def __init__(self, val_env, save_path, eval_freq=10_000, verbose=1):
        super().__init__(verbose)
        self.val_env     = val_env
        self.save_path   = save_path
        self.eval_freq   = eval_freq
        self.best_sharpe = -np.inf
        self.eval_history = []

    def _on_step(self) -> bool:
        if self.n_calls % self.eval_freq == 0:
            obs, _ = self.val_env.reset()
            daily_returns = []
            base_rewards = []
            rlhf_rewards = []
            done = False
            while not done:
                action, _ = self.model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = self.val_env.step(action)
                daily_returns.append(float(info.get('base_reward', reward)) / 1e-4)
                base_rewards.append(float(info.get('base_reward', reward)))
                rlhf_rewards.append(float(info.get('rlhf_reward', 0)))
                done = terminated or truncated

            if len(daily_returns) > 1:
                val_sharpe = sharpe_ratio(np.array(daily_returns))
                avg_rlhf = np.mean(rlhf_rewards)
                self.eval_history.append({
                    'step': self.num_timesteps,
                    'val_sharpe': val_sharpe,
                    'avg_base_reward': np.mean(base_rewards),
                    'avg_rlhf_reward': avg_rlhf,
                })
                if self.verbose:
                    print(
                        f'  [step {self.num_timesteps:>7,}] '
                        f'val Sharpe: {val_sharpe:.4f} '
                        f'(best: {self.best_sharpe:.4f}) '
                        f'| avg RLHF reward: {avg_rlhf:.4f}'
                    )
                if val_sharpe > self.best_sharpe:
                    self.best_sharpe = val_sharpe
                    self.model.save(self.save_path)
                    if self.verbose:
                        print(f'  → New best! Saved to {self.save_path}')
        return True

In [8]:
# ── RLHF Fine-tuning loop — 3 personas ──────────────────────────────────
rlhf_results = {}

for persona in personas:
    print(f'\n{"="*60}')
    print(f'RLHF fine-tuning: {persona}')
    print(f'{"="*60}')

    rm = reward_models[persona]

    # Build RLHF-wrapped train env
    train_env = make_env(
        df_train, mode='train',
        reward_model=rm,
        rlhf_lambda=RLHF_LAMBDA,
        seed=42,
    )

    # Build RLHF-wrapped val env (for evaluation)
    val_env = make_env(
        df_val, mode='val',
        reward_model=rm,
        rlhf_lambda=RLHF_LAMBDA,
        seed=42,
    )

    save_path = f'{CKPT_DIR}/rlhf_agent_{persona}'

    callback = RLHFEvalCallback(
        val_env   = val_env,
        save_path = save_path,
        eval_freq = EVAL_FREQ,
        verbose   = 1,
    )

    # Load base agent into RLHF env
    model = PPO.load(
        BASE_AGENT_PATH,
        env=train_env,
        device='cpu',
        tensorboard_log=f'{REPO_DIR}/runs/',
    )
    print(f'Loaded base agent from {BASE_AGENT_PATH}')
    print(f'Reward model: {persona} (lambda={RLHF_LAMBDA})')
    print(f'Training for {RLHF_TIMESTEPS:,} steps...')

    model.learn(
        total_timesteps=RLHF_TIMESTEPS,
        callback=callback,
        tb_log_name=f'rlhf_{persona}',
        reset_num_timesteps=True,
    )

    # Always save final model (callback only saves on best Sharpe)
    model.save(save_path)
    print(f'Saved final model → {save_path}.zip')

    rlhf_results[persona] = {
        'best_sharpe': callback.best_sharpe,
        'save_path': save_path + '.zip',
        'eval_history': callback.eval_history,
    }

    print(f'\n{persona} done. Best val Sharpe: {callback.best_sharpe:.4f}')
    print(f'Saved to: {save_path}.zip')

    train_env.close()
    val_env.close()


RLHF fine-tuning: conservative
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Loaded base agent from /content/drive/MyDrive/3001_RL_group_project/Project/results/checkpoints/base_agent_seed2.zip
Reward model: conservative (lambda=0.5)
Training for 300,000 steps...
Logging to /content/rlhf-portfolio/runs/rlhf_conservative_1


/content/rlhf-portfolio/src/metrics.py:25: RuntimeWarning: overflow encountered in scalar power
  return float(cumulative ** (TRADING_DAYS / n) - 1)
/content/rlhf-portfolio/src/metrics.py:25: RuntimeWarning: invalid value encountered in scalar power
  return float(cumulative ** (TRADING_DAYS / n) - 1)
/content/rlhf-portfolio/src/metrics.py:25: RuntimeWarning: overflow encountered in scalar power
  return float(cumulative ** (TRADING_DAYS / n) - 1)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/content/rlhf-portfolio/src/metrics.py:25: RuntimeWarning: invalid value encountered in scalar power
  return float(cumulative ** (TRADING_DAYS / n) - 1)


----------------------------------
| rollout/           |           |
|    ep_len_mean     | 2.01e+03  |
|    ep_rew_mean     | -1.05e+03 |
| time/              |           |
|    fps             | 133       |
|    iterations      | 1         |
|    time_elapsed    | 15        |
|    total_timesteps | 2048      |
----------------------------------


/content/rlhf-portfolio/src/metrics.py:25: RuntimeWarning: overflow encountered in scalar power
  return float(cumulative ** (TRADING_DAYS / n) - 1)
/content/rlhf-portfolio/src/metrics.py:25: RuntimeWarning: invalid value encountered in scalar power
  return float(cumulative ** (TRADING_DAYS / n) - 1)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 2.01e+03    |
|    ep_rew_mean          | -1.14e+03   |
| time/                   |             |
|    fps                  | 125         |
|    iterations           | 2           |
|    time_elapsed         | 32          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.027467377 |
|    clip_fraction        | 0.362       |
|    clip_range           | 0.2         |
|    entropy_loss         | -60.5       |
|    explained_variance   | -0.0252     |
|    learning_rate        | 0.0003      |
|    loss                 | 33.9        |
|    n_updates            | 1910        |
|    policy_gradient_loss | 0.00227     |
|    std                  | 1.83        |
|    value_loss           | 70.9        |
-----------------------------------------


...

  [step 300,000] val Sharpe: -0.9093 (best: 0.8316) | avg RLHF reward: -0.0446


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


day: 2013, episode: 150
begin_total_asset: 1000000.00
end_total_asset: -1740541.09
total_reward: -2740541.09
total_cost: 998.92
total_trades: 31010
Sharpe: 0.380


/content/rlhf-portfolio/src/metrics.py:25: RuntimeWarning: overflow encountered in scalar power
  return float(cumulative ** (TRADING_DAYS / n) - 1)
/content/rlhf-portfolio/src/metrics.py:25: RuntimeWarning: invalid value encountered in scalar power
  return float(cumulative ** (TRADING_DAYS / n) - 1)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 2.01e+03    |
|    ep_rew_mean          | -917        |
| time/                   |             |
|    fps                  | 104         |
|    iterations           | 147         |
|    time_elapsed         | 2892        |
|    total_timesteps      | 301056      |
| train/                  |             |
|    approx_kl            | 0.012143717 |
|    clip_fraction        | 0.229       |
|    clip_range           | 0.2         |
|    entropy_loss         | -72.7       |
|    explained_variance   | 1.17e-05    |
|    learning_rate        | 0.0003      |
|    loss                 | 155         |
|    n_updates            | 3360        |
|    policy_gradient_loss | 0.00522     |
|    std                  | 2.78        |
|    value_loss           | 258         |
-----------------------------------------
Saved final model → /content/drive/MyDrive/3001_RL_group_project/Project/res

In [9]:
# ── Summary ──────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('RLHF Fine-tuning — Summary')
print('='*60)

for persona in personas:
    r = rlhf_results[persona]
    print(f'  {persona:15s} best val Sharpe = {r["best_sharpe"]:.4f}  → {r["save_path"]}')

# Verify all checkpoints exist
print('\nCheckpoint verification:')
for persona in personas:
    path = f'{CKPT_DIR}/rlhf_agent_{persona}.zip'
    exists = os.path.exists(path)
    size = os.path.getsize(path) / 1e6 if exists else 0
    status = f'✓ {size:.1f} MB' if exists else '✗ MISSING'
    print(f'  {persona:15s} {status}')


RLHF Fine-tuning — Summary
  conservative    best val Sharpe = 1.2115  → /content/drive/MyDrive/3001_RL_group_project/Project/results/checkpoints/rlhf_agent_conservative.zip
  balanced        best val Sharpe = 0.9424  → /content/drive/MyDrive/3001_RL_group_project/Project/results/checkpoints/rlhf_agent_balanced.zip
  aggressive      best val Sharpe = 0.8316  → /content/drive/MyDrive/3001_RL_group_project/Project/results/checkpoints/rlhf_agent_aggressive.zip

Checkpoint verification:
  conservative    ✓ 3.9 MB
  balanced        ✓ 3.9 MB
  aggressive      ✓ 3.9 MB


In [10]:
# ── Plot training curves ─────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, persona in enumerate(personas):
    hist = rlhf_results[persona]['eval_history']
    if not hist:
        continue
    steps   = [h['step'] for h in hist]
    sharpes = [h['val_sharpe'] for h in hist]
    rlhf_r  = [h['avg_rlhf_reward'] for h in hist]

    ax = axes[i]
    ax2 = ax.twinx()

    ax.plot(steps, sharpes, 'b-o', markersize=3, label='Val Sharpe')
    ax2.plot(steps, rlhf_r, 'r--s', markersize=3, label='Avg RLHF reward')

    ax.set_xlabel('Training steps')
    ax.set_ylabel('Val Sharpe', color='b')
    ax2.set_ylabel('RLHF reward', color='r')
    ax.set_title(f'{persona.capitalize()} RLHF Agent')

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='lower right')

plt.tight_layout()
fig_dir = f'{DRIVE_PROJECT}/results/figures'
os.makedirs(fig_dir, exist_ok=True)
plt.savefig(f'{fig_dir}/rlhf_finetuning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {fig_dir}/rlhf_finetuning_curves.png')

Saved to /content/drive/MyDrive/3001_RL_group_project/Project/results/figures/rlhf_finetuning_curves.png
